# CLIP4Outfit — Notebook B2: pipeline de datos real + entrenamiento completo

Autocontenido: incluye de nuevo la arquitectura del Notebook B1 (WTI + estilo + outfit), ya
validada con las comprobaciones que hicimos, para no depender de ejecutar dos notebooks en el
mismo kernel. Conecta con:
- El CLIP fine-tuned del Notebook A (`clip_finetuned_polyvore.pt`).
- Los outfits reales de Polyvore (con la Capa 1 de filtrado ya aplicada, igual que en la
  Opción 1).

**Requisitos previos en disco:** `polyvore_outfit_titles.json`, `train/valid/test.json`
(partición `nondisjoint`), directorio de imágenes, y el checkpoint del Notebook A.


## 0. Imports y configuración

In [ ]:
import os, json, math, random, re
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from transformers import CLIPModel, CLIPTokenizerFast, get_cosine_schedule_with_warmup

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"  # AJUSTA A TU RUTA LOCAL
IMAGES_DIR    = POLYVORE_PATH + "images/"
OUTFIT_TITLES_PATH = POLYVORE_PATH + "polyvore_outfit_titles.json"

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_FINETUNED_CKPT = POLYVORE_PATH + "checkpoints_clip4outfit/clip_finetuned_polyvore.pt"
CLIP_IMG_SIZE = 224
MAX_TEXT_TOKENS = 24  # decisión propia -- los títulos de outfit son frases cortas, no hace falta el máximo de CLIP (77)
D_MODEL = 512

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Arquitectura de CLIP4Outfit (idéntica al Notebook B1, ya validada)

In [ ]:
P_WEIGHT = 0.2
ALPHA_STYLE = 0.3
BETA_OUTFIT = 0.3
KO_RATIO = 1 / 3
KT_RATIO = 1 / 6
TAU_SCALE = 100.0


def pad_and_mask(tensor_list):
    B = len(tensor_list)
    D = tensor_list[0].shape[-1]
    n_max = max(t.shape[0] for t in tensor_list)
    device_ = tensor_list[0].device
    dtype_ = tensor_list[0].dtype
    padded = torch.zeros(B, n_max, D, device=device_, dtype=dtype_)
    mask = torch.zeros(B, n_max, dtype=torch.bool, device=device_)
    for i, t in enumerate(tensor_list):
        n = t.shape[0]
        padded[i, :n] = t
        mask[i, :n] = True
    return padded, mask


def pad_and_mask_images(image_list):
    """Igual que pad_and_mask pero para tensores de imagen (N, C, H, W) en vez de (N, D)."""
    B = len(image_list)
    C, H, W = image_list[0].shape[1:]
    n_max = max(t.shape[0] for t in image_list)
    device_ = image_list[0].device
    padded = torch.zeros(B, n_max, C, H, W, device=device_)
    mask = torch.zeros(B, n_max, dtype=torch.bool, device=device_)
    for i, t in enumerate(image_list):
        n = t.shape[0]
        padded[i, :n] = t
        mask[i, :n] = True
    return padded, mask


def masked_softmax(logits, mask, dim=-1):
    logits = logits.masked_fill(~mask, float("-inf"))
    probs = F.softmax(logits, dim=dim)
    return torch.nan_to_num(probs, nan=0.0)


class AdaptiveWeightNet(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.linear = nn.Linear(d_model, 1)

    def forward(self, x, mask):
        logits = self.linear(x).squeeze(-1)
        return masked_softmax(logits, mask, dim=-1)


def wti_similarity_batch(item_emb, item_mask, text_emb, text_mask, fo_weight, ft_weight, p=P_WEIGHT):
    sim = torch.einsum("nid,mjd->nmij", item_emb, text_emb)
    joint_mask = item_mask.unsqueeze(1).unsqueeze(-1) & text_mask.unsqueeze(0).unsqueeze(2)
    sim = sim.masked_fill(~joint_mask, float("-inf"))

    max_over_tokens = sim.max(dim=3).values
    max_over_tokens = torch.nan_to_num(max_over_tokens, neginf=0.0)
    term1 = (fo_weight.unsqueeze(1) * max_over_tokens).sum(dim=2)

    max_over_items = sim.max(dim=2).values
    max_over_items = torch.nan_to_num(max_over_items, neginf=0.0)
    term2 = (ft_weight.unsqueeze(0) * max_over_items).sum(dim=2)

    return (term1 + p * term2) / (1 + p)


class StyleTransformer(nn.Module):
    def __init__(self, d_model, n_heads=8, ffn_dim=None, dropout=0.1):
        super().__init__()
        ffn_dim = ffn_dim or d_model * 4
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x


@torch.no_grad()
def greedy_kmeans_init(X, k):
    N = X.shape[0]
    dists = torch.cdist(X, X)
    first_idx = dists.sum(dim=1).argmax().item()
    chosen = [first_idx]
    for _ in range(1, k):
        centroid_stack = X[chosen]
        d_to_centroids = torch.cdist(X, centroid_stack)
        min_d = d_to_centroids.min(dim=1).values
        min_d[chosen] = -float("inf")
        chosen.append(min_d.argmax().item())
    return X[chosen].clone()


@torch.no_grad()
def kmeans_assign(X, k, n_iters=5):
    N = X.shape[0]
    centroids = greedy_kmeans_init(X, k)
    assign = torch.zeros(N, dtype=torch.long, device=X.device)
    for _ in range(n_iters):
        d = torch.cdist(X, centroids)
        assign = d.argmin(dim=1)
        for c in range(k):
            members = X[assign == c]
            if members.shape[0] > 0:
                centroids[c] = members.mean(dim=0)
    return assign


def style_extractor(X, k_ratio):
    N = X.shape[0]
    if N <= 1:
        return X
    k = max(1, int(k_ratio * N))
    k = min(k, N)
    assign = kmeans_assign(X.detach(), k)
    centroids = []
    for c in range(k):
        members = X[assign == c]
        if members.shape[0] > 0:
            centroids.append(members.mean(dim=0))
    if not centroids:
        centroids = [X.mean(dim=0)]
    out = torch.stack(centroids)
    return F.normalize(out, p=2, dim=-1)


class OutfitLevelEncoder(nn.Module):
    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__()
        self.transformer = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)

    def forward(self, x, key_padding_mask=None):
        x = self.transformer(x, key_padding_mask=key_padding_mask)
        real_mask = (~key_padding_mask).unsqueeze(-1).float() if key_padding_mask is not None \
            else torch.ones(x.shape[0], x.shape[1], 1, device=x.device)
        pooled = (x * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1.0)
        return F.normalize(pooled, p=2, dim=-1)


class CLIP4OutfitCore(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=8, dropout=0.1,
                 ko_ratio=KO_RATIO, kt_ratio=KT_RATIO, p_weight=P_WEIGHT):
        super().__init__()
        self.ko_ratio = ko_ratio
        self.kt_ratio = kt_ratio
        self.p_weight = p_weight

        self.fo_item = AdaptiveWeightNet(d_model)
        self.ft_item = AdaptiveWeightNet(d_model)

        self.style_transformer_o = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.style_transformer_t = StyleTransformer(d_model, n_heads=n_heads, dropout=dropout)
        self.fo_style = AdaptiveWeightNet(d_model)
        self.ft_style = AdaptiveWeightNet(d_model)

        self.outfit_encoder_o = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)
        self.outfit_encoder_t = OutfitLevelEncoder(d_model, n_heads=n_heads, dropout=dropout)

    def forward(self, item_emb, item_mask, text_emb, text_mask):
        B = item_emb.shape[0]

        fo_i = self.fo_item(item_emb, item_mask)
        ft_i = self.ft_item(text_emb, text_mask)
        S_item = wti_similarity_batch(item_emb, item_mask, text_emb, text_mask, fo_i, ft_i, p=self.p_weight)

        e_prime_o = self.style_transformer_o(item_emb, key_padding_mask=~item_mask)
        e_prime_t = self.style_transformer_t(text_emb, key_padding_mask=~text_mask)

        style_o_list, style_t_list = [], []
        for b in range(B):
            valid_o = e_prime_o[b][item_mask[b]]
            valid_t = e_prime_t[b][text_mask[b]]
            style_o_list.append(style_extractor(valid_o, self.ko_ratio))
            style_t_list.append(style_extractor(valid_t, self.kt_ratio))

        style_o, style_o_mask = pad_and_mask(style_o_list)
        style_t, style_t_mask = pad_and_mask(style_t_list)

        fo_s = self.fo_style(style_o, style_o_mask)
        ft_s = self.ft_style(style_t, style_t_mask)
        S_style = wti_similarity_batch(style_o, style_o_mask, style_t, style_t_mask, fo_s, ft_s, p=self.p_weight)

        o_vec = self.outfit_encoder_o(style_o, key_padding_mask=~style_o_mask)
        t_vec = self.outfit_encoder_t(style_t, key_padding_mask=~style_t_mask)
        S_outfit = o_vec @ t_vec.T

        return S_item, S_style, S_outfit


def masked_nce(logits, positive_mask):
    log_prob = F.log_softmax(logits, dim=1)
    pos_count = positive_mask.sum(dim=1).clamp(min=1.0)
    loss_per_row = -(log_prob * positive_mask).sum(dim=1) / pos_count
    return loss_per_row.mean()


def level_loss(S, group_ids, tau_scale=TAU_SCALE):
    logits = S * tau_scale
    positive_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0)).float()
    loss_o2t = masked_nce(logits, positive_mask)
    loss_t2o = masked_nce(logits.T, positive_mask.T)
    return (loss_o2t + loss_t2o) / 2


def clip4outfit_loss(S_item, S_style, S_outfit, group_ids, alpha=ALPHA_STYLE, beta=BETA_OUTFIT):
    L_item = level_loss(S_item, group_ids)
    L_style = level_loss(S_style, group_ids)
    L_outfit = level_loss(S_outfit, group_ids)
    L_total = L_item + alpha * L_style + beta * L_outfit
    return L_total, {"L_item": L_item.item(), "L_style": L_style.item(),
                      "L_outfit": L_outfit.item(), "L_total": L_total.item()}


print("Arquitectura cargada.")

## 2. Cargar el CLIP fine-tuned (Notebook A) + tokenizer

In [ ]:
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(device)
tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME)

ckpt_clip = torch.load(CLIP_FINETUNED_CKPT, map_location=device)
clip_model.load_state_dict(ckpt_clip["clip_state_dict"])
print(f"CLIP fine-tuned cargado (época {ckpt_clip['epoch']}, val_loss={ckpt_clip['val_loss']:.4f})")

core_model = CLIP4OutfitCore().to(device)
n_params_core = sum(p.numel() for p in core_model.parameters())
n_params_clip = sum(p.numel() for p in clip_model.parameters())
print(f"Parámetros CLIP4OutfitCore (nuevos): {n_params_core:,}")
print(f"Parámetros CLIP (backbone, también entrenable): {n_params_clip:,}")

## 3. Construir pares (outfit, texto) -- reutilizando la Capa 1 (filtro de spam/afiliación)

Mismo criterio que en la Opción 1: `title` con prioridad, `url_name` como *fallback* (cobertura
100%), y descarte de textos de spam/afiliación. **No aplicamos aquí la Capa 4** (cap de
frecuencia) -- recordemos que en la Opción 1 vimos que, una vez corregido el multi-positivo, el
cap ya no aportaba nada y solo restaba datos; aquí ya nace con multi-positivo desde el principio,
así que no hay motivo para aplicarlo.

In [ ]:
def build_outfit_item_map(json_paths_with_split):
    outfit_items = {}
    outfit_split = {}
    for path, split in json_paths_with_split:
        with open(path) as f:
            outfits = json.load(f)
        for outfit in outfits:
            set_id = str(outfit["set_id"])
            items_sorted = sorted(outfit["items"], key=lambda it: it["index"])
            outfit_items[set_id] = [str(it["item_id"]) for it in items_sorted]
            outfit_split[set_id] = split
    return outfit_items, outfit_split


OUTFIT_SPLIT_PATHS = [
    (POLYVORE_PATH + "nondisjoint/train.json", "train"),
    (POLYVORE_PATH + "nondisjoint/valid.json", "valid"),
    (POLYVORE_PATH + "nondisjoint/test.json",  "test"),
]
outfit_items, outfit_split = build_outfit_item_map(OUTFIT_SPLIT_PATHS)

with open(OUTFIT_TITLES_PATH) as f:
    outfit_titles_raw = json.load(f)


def clean_text_basic(s):
    return " ".join(s.strip().split())


def get_outfit_text(entry):
    title = clean_text_basic(entry.get("title", ""))
    if title:
        return title
    return clean_text_basic(entry.get("url_name", ""))


STORE_PATTERN = re.compile(
    r"\b(yesstyle|banggood|romwe|shein|zaful|aliexpress|paoloshoes|dhgate|rosegal|wish)\b", re.I)
PROMO_PATTERN = re.compile(r"\b(off|coupon|code|sale|discount|deal|free shipping)\b", re.I)
SIZE_PATTERN  = re.compile(r"\b(plus size|size \d|us \d|eu \d)\b", re.I)


def is_spam_text(text):
    return bool(STORE_PATTERN.search(text) or PROMO_PATTERN.search(text) or SIZE_PATTERN.search(text))


MAX_ITEMS_PER_OUTFIT = 19  # rango real de Polyvore Outfit (Sección 4.2 del paper de CLIP4Outfit)

# Reutilizamos glove_text_features.csv (ya generado por tu pipeline original) como lista
# vetada de item_id con imagen existente -- evita miles de comprobaciones de disco (os.path.isfile)
# una por una, mucho más rápido.
import pandas as pd
df_valid_items = pd.read_csv(POLYVORE_PATH + "data/processed/glove_text_features.csv")
valid_item_ids_with_image = set(df_valid_items["item_id"].astype(str))
print(f"Ítems con imagen vetada disponible: {len(valid_item_ids_with_image)}")

t2o_samples = []
skipped_no_text = skipped_spam = skipped_no_items = skipped_too_long = skipped_missing_img = 0

for set_id, entry in outfit_titles_raw.items():
    text = get_outfit_text(entry)
    if not text:
        skipped_no_text += 1
        continue
    if is_spam_text(text):
        skipped_spam += 1
        continue
    item_ids = outfit_items.get(set_id)
    if not item_ids:
        skipped_no_items += 1
        continue
    if len(item_ids) > MAX_ITEMS_PER_OUTFIT:
        skipped_too_long += 1
        continue
    if not all(iid in valid_item_ids_with_image for iid in item_ids):
        skipped_missing_img += 1
        continue

    t2o_samples.append({
        "set_id": set_id,
        "item_ids": item_ids,
        "text": text,
        "split": outfit_split.get(set_id, "unknown"),
    })

print(f"Muestras utilizables: {len(t2o_samples)}")
print(f"  Sin texto: {skipped_no_text}  |  Spam: {skipped_spam}  |  "
      f"Sin outfit en splits: {skipped_no_items}  |  >19 prendas: {skipped_too_long}  |  "
      f"Imagen faltante: {skipped_missing_img}")

t2o_train = [s for s in t2o_samples if s["split"] == "train"]
t2o_valid = [s for s in t2o_samples if s["split"] == "valid"]
t2o_test  = [s for s in t2o_samples if s["split"] == "test"]
print(f"train={len(t2o_train)}  valid={len(t2o_valid)}  test={len(t2o_test)}")

## 4. Preprocesado de imagen (idéntico al Notebook A) + `Dataset`/`DataLoader`

Cada outfit tiene un número distinto de prendas (`No` variable) -- el `collate_fn` las
rellena hasta la más larga del batch y construye la máscara correspondiente, igual que en el
Notebook B1 pero ahora con imágenes reales en vez de vectores sintéticos.

In [ ]:
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

clip_image_transform = transforms.Compose([
    transforms.Resize(CLIP_IMG_SIZE, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CLIP_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])


def load_clip_image(path):
    img = Image.open(path).convert("RGB")
    return clip_image_transform(img)


class OutfitTextDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        images = torch.stack([load_clip_image(os.path.join(IMAGES_DIR, f"{iid}.jpg"))
                               for iid in s["item_ids"]])
        return images, s["text"], s["set_id"]


class OutfitCollateFn:
    """Clase (no función anidada) para que sea picklable -- necesario en Windows si más
    adelante activas num_workers>0 (ver la lección de la Opción 1)."""
    def __init__(self, tokenizer, max_tokens):
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens

    def __call__(self, batch):
        images_list, texts, set_ids = zip(*batch)
        item_images, item_mask = pad_and_mask_images(list(images_list))
        tok = self.tokenizer(list(texts), padding="max_length", truncation=True,
                              max_length=self.max_tokens, return_tensors="pt")
        return item_images, item_mask, tok["input_ids"], tok["attention_mask"], list(texts), list(set_ids)


collate_fn = OutfitCollateFn(tokenizer, MAX_TEXT_TOKENS)

# NUM_WORKERS=0 por la misma razón que en el Notebook A -- súbelo con cuidado si quieres
# recuperar velocidad, ya usamos una clase picklable en vez de una clausura.
NUM_WORKERS = 0

train_dataset = OutfitTextDataset(t2o_train)
valid_dataset = OutfitTextDataset(t2o_valid)
test_dataset  = OutfitTextDataset(t2o_test)

print(f"Datasets listos: train={len(train_dataset)}  valid={len(valid_dataset)}  test={len(test_dataset)}")

## 5. Funciones de extracción de embeddings desde CLIP -- CON gradiente

Esto es lo que discutimos: **no** se usa `get_image_features`/`get_text_features` sin más
para el texto (eso daría un único vector por frase, perdiendo la granularidad por palabra que
exige la WTI). Para las imágenes de ítem sí es correcto usar `get_image_features` (cada prenda
es una unidad completa que se resume en un vector, tal como especifica el paper).

In [ ]:
def extract_image_embeds(output):
    """Robusto a la versión de transformers instalada: en algunas, get_image_features
    devuelve directamente el tensor de embeddings; en otras (más recientes, ej. 5.x),
    devuelve un objeto envoltorio cuyo .pooler_output / .image_embeds YA es el embedding
    final proyectado (comprobado empíricamente: NO hay que volver a aplicar visual_projection
    encima, ya viene aplicado)."""
    if torch.is_tensor(output):
        return output
    if getattr(output, "image_embeds", None) is not None:
        return output.image_embeds
    if getattr(output, "pooler_output", None) is not None:
        return output.pooler_output
    raise TypeError(f"No se pudo extraer el tensor de embeddings de imagen de {type(output)}")


def encode_items(clip_model, item_images, item_mask):
    """item_images: (B, No_max, 3, H, W). item_mask: (B, No_max) bool.
    Devuelve item_emb: (B, No_max, D) L2-normalizado (posiciones de padding puestas a 0)."""
    B, No_max = item_images.shape[:2]
    flat = item_images.reshape(B * No_max, 3, CLIP_IMG_SIZE, CLIP_IMG_SIZE)
    embeds = extract_image_embeds(clip_model.get_image_features(pixel_values=flat))  # (B*No_max, D)
    embeds = F.normalize(embeds, p=2, dim=-1)
    embeds = embeds.reshape(B, No_max, -1)
    embeds = embeds * item_mask.unsqueeze(-1)  # limpieza -- no es estrictamente necesario, ya se enmascara después
    return embeds


def encode_text_tokens(clip_model, input_ids, attention_mask):
    """input_ids, attention_mask: (B, Nt_max). Devuelve token_embeds: (B, Nt_max, D)
    L2-normalizado (uno por PALABRA/token, no un único vector por frase) y text_mask (B, Nt_max)."""
    outputs = clip_model.text_model(input_ids=input_ids, attention_mask=attention_mask)
    token_hidden = outputs.last_hidden_state              # (B, Nt_max, hidden_size)
    token_embeds = clip_model.text_projection(token_hidden)  # (B, Nt_max, D) -- proyección lineal, funciona posición a posición
    token_embeds = F.normalize(token_embeds, p=2, dim=-1)
    text_mask = attention_mask.bool()
    token_embeds = token_embeds * text_mask.unsqueeze(-1)
    return token_embeds, text_mask

## 6. Encontrar el *micro-batch* de outfits máximo para tu GPU

Aquí la VRAM depende del número TOTAL de imágenes procesadas a la vez (outfits × prendas por
outfit, hasta 19 cada uno) -- no solo del número de outfits. Usamos outfits sintéticos con
`No` variable (muestreado del rango real, 4-19) para medir el pico de memoria de verdad,
igual que hicimos en el Notebook A pero adaptado a esta estructura.

In [ ]:
# def find_max_micro_batch_outfits(clip_model, core_model, tokenizer, device,
#                                    no_range=(4, 19), start_outfits=1, max_outfits=32):
#     clip_model.train()
#     core_model.train()

#     def try_micro_batch(n_outfits):
#         try:
#             torch.cuda.empty_cache()
#             torch.cuda.reset_peak_memory_stats()

#             nos = [random.randint(*no_range) for _ in range(n_outfits)]
#             no_max = max(nos)
#             item_images = torch.randn(n_outfits, no_max, 3, CLIP_IMG_SIZE, CLIP_IMG_SIZE, device=device)
#             item_mask = torch.zeros(n_outfits, no_max, dtype=torch.bool, device=device)
#             for i, n in enumerate(nos):
#                 item_mask[i, :n] = True

#             input_ids = torch.randint(0, tokenizer.vocab_size, (n_outfits, MAX_TEXT_TOKENS), device=device)
#             attention_mask = torch.ones(n_outfits, MAX_TEXT_TOKENS, dtype=torch.long, device=device)
#             group_ids = torch.arange(n_outfits, device=device)

#             with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
#                 item_emb = encode_items(clip_model, item_images, item_mask)
#                 text_emb, text_mask = encode_text_tokens(clip_model, input_ids, attention_mask)
#                 S_i, S_s, S_o = core_model(item_emb, item_mask, text_emb, text_mask)
#                 loss, _ = clip4outfit_loss(S_i, S_s, S_o, group_ids)
#             loss.backward()
#             clip_model.zero_grad()
#             core_model.zero_grad()
#             peak_gb = torch.cuda.max_memory_allocated() / 1e9
#             return True, peak_gb
#         except torch.cuda.OutOfMemoryError:
#             torch.cuda.empty_cache()
#             return False, None

#     n = start_outfits
#     last_ok = None
#     while n <= max_outfits:
#         ok, peak_gb = try_micro_batch(n)
#         print(f"outfits={n:3d} -> {'OK' if ok else 'OOM'}" + (f"  (pico: {peak_gb:.2f} GB)" if ok else ""))
#         if not ok:
#             break
#         last_ok = n
#         n *= 2

#     if last_ok is None:
#         print("Ni 1 outfit cabe de una vez -- revisa MAX_ITEMS_PER_OUTFIT o considera reducir la resolución de imagen.")
#         return start_outfits

#     low, high = last_ok, n
#     while high - low > 1:
#         mid = (low + high) // 2
#         ok, peak_gb = try_micro_batch(mid)
#         print(f"outfits={mid:3d} -> {'OK' if ok else 'OOM'}" + (f"  (pico: {peak_gb:.2f} GB)" if ok else ""))
#         if ok:
#             low = mid
#         else:
#             high = mid

#     safe = max(1, int(low * 0.85))
#     print(f"\nMáximo estable: {low} outfits/micro-batch. Recomendado con margen: {safe}")
#     return safe


# if device.type == "cuda":
#     MICRO_BATCH_OUTFITS = find_max_micro_batch_outfits(clip_model, core_model, tokenizer, device)
# else:
#     MICRO_BATCH_OUTFITS = 1
#     print("Sin GPU -- usando micro-batch de 1 para CPU (será muy lento).")

MICRO_BATCH_OUTFITS = 4

TARGET_EFFECTIVE_BATCH = 50  # batch efectivo del paper (Sección 4.2)
GRAD_ACCUM_STEPS = max(1, TARGET_EFFECTIVE_BATCH // MICRO_BATCH_OUTFITS)
print(f"\nMICRO_BATCH_OUTFITS={MICRO_BATCH_OUTFITS}  GRAD_ACCUM_STEPS={GRAD_ACCUM_STEPS}  "
      f"-> batch efectivo real={MICRO_BATCH_OUTFITS * GRAD_ACCUM_STEPS}")

train_loader = DataLoader(train_dataset, batch_size=MICRO_BATCH_OUTFITS, shuffle=True,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS,
                           pin_memory=(device.type == "cuda"))
valid_loader = DataLoader(valid_dataset, batch_size=MICRO_BATCH_OUTFITS, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS,
                           pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_dataset, batch_size=MICRO_BATCH_OUTFITS, shuffle=False,
                           collate_fn=collate_fn, num_workers=NUM_WORKERS,
                           pin_memory=(device.type == "cuda"))



## 7. Evaluación: Recall@k multi-positivo, usando `S_outfit` como puntuación de recuperación

`S_outfit` es la similitud del nivel más agregado (el vector global de outfit contra el vector
global de texto) -- es la puntuación final natural para ranking en el modelo completo
(I+S+O). Para la futura ablación (Tabla 2 del paper: I / I+S / I+S+O) bastará con sumar los
niveles que correspondan en cada variante; dejo `S_item`/`S_style` accesibles para eso.

In [ ]:
@torch.no_grad()
def evaluate_recall_at_k(loader, ks=(1, 5, 10, 30, 50)):
    clip_model.eval()
    core_model.eval()

    all_S_outfit_rows = []
    all_group_ids = []
    all_texts_seen = []

    # Primero, extraemos embeddings de TODOS los outfits/textos del split (sin cruzar todavía)
    all_item_embs, all_item_masks, all_text_embs, all_text_masks, all_texts = [], [], [], [], []
    for item_images, item_mask, input_ids, attention_mask, texts, set_ids in loader:
        item_images = item_images.to(device)
        item_mask = item_mask.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            item_emb = encode_items(clip_model, item_images, item_mask)
            text_emb, text_mask = encode_text_tokens(clip_model, input_ids, attention_mask)

        all_item_embs.append(item_emb.cpu())
        all_item_masks.append(item_mask.cpu())
        all_text_embs.append(text_emb.cpu())
        all_text_masks.append(text_mask.cpu())
        all_texts.extend(texts)

    # group_ids sobre TODO el split (mismo texto = mismo grupo, multi-positivo)
    text_to_gid = {}
    group_ids = []
    for t in all_texts:
        if t not in text_to_gid:
            text_to_gid[t] = len(text_to_gid)
        group_ids.append(text_to_gid[t])
    group_ids = torch.tensor(group_ids)

    # Recorremos por bloques para no construir una matriz N x N con imágenes reales de golpe en VRAM
    # -- aquí basta con los vectores ya extraídos (mucho más ligero que el forward completo).
    N = len(all_texts)
    n_max_item = max(t.shape[1] for t in all_item_embs)
    n_max_text = max(t.shape[1] for t in all_text_embs)

    def repad(tensor_list, mask_list, n_max):
        out, out_mask = [], []
        for t, m in zip(tensor_list, mask_list):
            b, n, d = t.shape
            if n < n_max:
                pad_t = torch.zeros(b, n_max - n, d)
                pad_m = torch.zeros(b, n_max - n, dtype=torch.bool)
                t = torch.cat([t, pad_t], dim=1)
                m = torch.cat([m, pad_m], dim=1)
            out.append(t)
            out_mask.append(m)
        return torch.cat(out, dim=0), torch.cat(out_mask, dim=0)

    item_emb_all, item_mask_all = repad(all_item_embs, all_item_masks, n_max_item)
    text_emb_all, text_mask_all = repad(all_text_embs, all_text_masks, n_max_text)

    item_emb_all, item_mask_all = item_emb_all.to(device), item_mask_all.to(device)
    text_emb_all, text_mask_all = text_emb_all.to(device), text_mask_all.to(device)

    # S_outfit para TODO el split de una vez -- son vectores (N, D) ya agregados, no imágenes,
    # así que esto sí cabe en VRAM aunque N sea miles.
    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        fo_i = core_model.fo_item(item_emb_all, item_mask_all)
        ft_i = core_model.ft_item(text_emb_all, text_mask_all)
        e_prime_o = core_model.style_transformer_o(item_emb_all, key_padding_mask=~item_mask_all)
        e_prime_t = core_model.style_transformer_t(text_emb_all, key_padding_mask=~text_mask_all)

        style_o_list, style_t_list = [], []
        for b in range(N):
            valid_o = e_prime_o[b][item_mask_all[b]]
            valid_t = e_prime_t[b][text_mask_all[b]]
            style_o_list.append(style_extractor(valid_o, core_model.ko_ratio))
            style_t_list.append(style_extractor(valid_t, core_model.kt_ratio))

        style_o, style_o_mask = pad_and_mask(style_o_list)
        style_t, style_t_mask = pad_and_mask(style_t_list)

        o_vec = core_model.outfit_encoder_o(style_o, key_padding_mask=~style_o_mask)
        t_vec = core_model.outfit_encoder_t(style_t, key_padding_mask=~style_t_mask)
        S_outfit_full = o_vec @ t_vec.T  # (N, N)

    group_ids = group_ids.to(device)
    relevant_mask = (group_ids.unsqueeze(1) == group_ids.unsqueeze(0))

    sorted_idx = S_outfit_full.argsort(dim=1, descending=True)
    ranks_matrix = torch.empty_like(sorted_idx)
    rank_values = torch.arange(1, N + 1, device=device).unsqueeze(0).expand(N, -1)
    ranks_matrix.scatter_(1, sorted_idx, rank_values)

    ranks_masked = torch.where(relevant_mask, ranks_matrix, torch.full_like(ranks_matrix, N + 1))
    best_rank = ranks_masked.min(dim=1).values.float()

    results = {f"Recall@{k}": (best_rank <= k).float().mean().item() for k in ks}
    results["MRR"] = (1.0 / best_rank).mean().item()
    results["MedianRank"] = best_rank.median().item()

    clip_model.train()
    core_model.train()
    return results

In [ ]:
@torch.no_grad()
def evaluate_recall_at_k_strict(loader, ks=(1, 5, 10, 30, 50)):
    """Igual que evaluate_recall_at_k, pero con criterio de UN SOLO positivo (el set_id
    exacto), no multi-positivo -- para poder comparar contra los números que reporta el
    paper con la vara de medir más probable que usaron ellos."""
    clip_model.eval()
    core_model.eval()

    all_item_embs, all_item_masks, all_text_embs, all_text_masks = [], [], [], []
    all_set_ids = []
    for item_images, item_mask, input_ids, attention_mask, texts, set_ids in loader:
        item_images = item_images.to(device)
        item_mask = item_mask.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            item_emb = encode_items(clip_model, item_images, item_mask)
            text_emb, text_mask = encode_text_tokens(clip_model, input_ids, attention_mask)

        all_item_embs.append(item_emb.cpu())
        all_item_masks.append(item_mask.cpu())
        all_text_embs.append(text_emb.cpu())
        all_text_masks.append(text_mask.cpu())
        all_set_ids.extend(set_ids)

    N = len(all_set_ids)
    n_max_item = max(t.shape[1] for t in all_item_embs)
    n_max_text = max(t.shape[1] for t in all_text_embs)

    def repad(tensor_list, mask_list, n_max):
        out, out_mask = [], []
        for t, m in zip(tensor_list, mask_list):
            b, n, d = t.shape
            if n < n_max:
                pad_t = torch.zeros(b, n_max - n, d)
                pad_m = torch.zeros(b, n_max - n, dtype=torch.bool)
                t = torch.cat([t, pad_t], dim=1)
                m = torch.cat([m, pad_m], dim=1)
            out.append(t)
            out_mask.append(m)
        return torch.cat(out, dim=0), torch.cat(out_mask, dim=0)

    item_emb_all, item_mask_all = repad(all_item_embs, all_item_masks, n_max_item)
    text_emb_all, text_mask_all = repad(all_text_embs, all_text_masks, n_max_text)
    item_emb_all, item_mask_all = item_emb_all.to(device), item_mask_all.to(device)
    text_emb_all, text_mask_all = text_emb_all.to(device), text_mask_all.to(device)

    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        e_prime_o = core_model.style_transformer_o(item_emb_all, key_padding_mask=~item_mask_all)
        e_prime_t = core_model.style_transformer_t(text_emb_all, key_padding_mask=~text_mask_all)

        style_o_list, style_t_list = [], []
        for b in range(N):
            valid_o = e_prime_o[b][item_mask_all[b]]
            valid_t = e_prime_t[b][text_mask_all[b]]
            style_o_list.append(style_extractor(valid_o, core_model.ko_ratio))
            style_t_list.append(style_extractor(valid_t, core_model.kt_ratio))

        style_o, style_o_mask = pad_and_mask(style_o_list)
        style_t, style_t_mask = pad_and_mask(style_t_list)

        o_vec = core_model.outfit_encoder_o(style_o, key_padding_mask=~style_o_mask)
        t_vec = core_model.outfit_encoder_t(style_t, key_padding_mask=~style_t_mask)
        S_outfit_full = o_vec @ t_vec.T  # (N, N)

    # -- ÚNICA diferencia real respecto a evaluate_recall_at_k: el objetivo es SOLO la
    # posición diagonal (mismo índice = mismo set_id), no todo el grupo de mismo texto.
    sorted_idx = S_outfit_full.argsort(dim=1, descending=True)
    ranks_matrix = torch.empty_like(sorted_idx)
    rank_values = torch.arange(1, N + 1, device=device).unsqueeze(0).expand(N, -1)
    ranks_matrix.scatter_(1, sorted_idx, rank_values)

    target_rank = ranks_matrix.diagonal().float()  # rank de la propia posición, para cada fila

    results = {f"Recall@{k}": (target_rank <= k).float().mean().item() for k in ks}
    results["MRR"] = (1.0 / target_rank).mean().item()
    results["MedianRank"] = target_rank.median().item()

    clip_model.train()
    core_model.train()
    return results

## 8. Bucle de entrenamiento -- hiperparámetros exactos del paper (Sección 4.2)

`Adam` (no `AdamW`, tal como especifica el paper), `LR=5×10⁻⁷`, *cosine annealing* sobre **20
épocas** en total (el *schedule* se calcula sobre el número real de pasos de optimización, no
de *forwards* -- importante con `GRAD_ACCUM_STEPS>1`), batch efectivo 50 vía *gradient
accumulation*.

In [ ]:
def gradcache_step(clip_model, core_model, micro_batches, device):
    """micro_batches: lista de tuplas (item_images, item_mask, input_ids, attention_mask, texts)
    -- todos los micro-batches que forman UN batch efectivo (los GRAD_ACCUM_STEPS acumulados).
    Rellena .grad en clip_model y core_model como si se hubiera hecho un único forward/backward
    sobre el batch efectivo completo (negativos reales de 50, no de MICRO_BATCH_OUTFITS)."""

    # ---------- FASE 1: representaciones SIN gradiente (barato en memoria) ----------
    rng_states = []
    item_embs_detached, item_masks, text_embs_detached, text_masks = [], [], [], []
    all_texts = []

    with torch.no_grad():
        for item_images, item_mask, input_ids, attention_mask, texts in micro_batches:
            rng_states.append(torch.random.get_rng_state())  # salvaguarda por si activas dropout en CLIP

            item_images_d = item_images.to(device)
            item_mask_d = item_mask.to(device)
            input_ids_d = input_ids.to(device)
            attention_mask_d = attention_mask.to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                item_emb = encode_items(clip_model, item_images_d, item_mask_d)
                text_emb, text_mask_d = encode_text_tokens(clip_model, input_ids_d, attention_mask_d)

            item_embs_detached.append(item_emb)
            item_masks.append(item_mask_d)
            text_embs_detached.append(text_emb)
            text_masks.append(text_mask_d)
            all_texts.extend(texts)

    def repad(tensor_list, mask_list, n_max):
        out_t, out_m = [], []
        for t, m in zip(tensor_list, mask_list):
            b, n, d = t.shape
            if n < n_max:
                pad_t = torch.zeros(b, n_max - n, d, device=t.device, dtype=t.dtype)
                pad_m = torch.zeros(b, n_max - n, dtype=torch.bool, device=m.device)
                t = torch.cat([t, pad_t], dim=1)
                m = torch.cat([m, pad_m], dim=1)
            out_t.append(t)
            out_m.append(m)
        return torch.cat(out_t, dim=0), torch.cat(out_m, dim=0)

    n_max_item = max(t.shape[1] for t in item_embs_detached)
    n_max_text = max(t.shape[1] for t in text_embs_detached)
    item_emb_full, item_mask_full = repad(item_embs_detached, item_masks, n_max_item)
    text_emb_full, text_mask_full = repad(text_embs_detached, text_masks, n_max_text)

    item_emb_full = item_emb_full.clone().requires_grad_(True)
    text_emb_full = text_emb_full.clone().requires_grad_(True)

    text_to_gid = {}
    group_ids = []
    for t in all_texts:
        if t not in text_to_gid:
            text_to_gid[t] = len(text_to_gid)
        group_ids.append(text_to_gid[t])
    group_ids = torch.tensor(group_ids, device=device)

    # ---------- FASE 2: pérdida real sobre el BATCH EFECTIVO COMPLETO (negativos de verdad) ----------
    with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
        S_i, S_s, S_o = core_model(item_emb_full, item_mask_full, text_emb_full, text_mask_full)
        loss, loss_parts = clip4outfit_loss(S_i, S_s, S_o, group_ids)

    scaler.scale(loss).backward()  # rellena .grad de item_emb_full/text_emb_full Y de core_model directamente

    item_grad_full = item_emb_full.grad.detach()
    text_grad_full = text_emb_full.grad.detach()

    # ---------- FASE 3: repropagar por cada micro-batch, esta vez CON gradiente en CLIP ----------
    offset = 0
    for mb_idx, (item_images, item_mask, input_ids, attention_mask, texts) in enumerate(micro_batches):
        b = item_images.shape[0]
        torch.random.set_rng_state(rng_states[mb_idx])  # misma máscara de dropout que en la Fase 1 (si la hubiera)

        item_images_d = item_images.to(device)
        item_mask_d = item_mask.to(device)
        input_ids_d = input_ids.to(device)
        attention_mask_d = attention_mask.to(device)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            item_emb_mb = encode_items(clip_model, item_images_d, item_mask_d)       # CON gradiente esta vez
            text_emb_mb, _ = encode_text_tokens(clip_model, input_ids_d, attention_mask_d)

        g_item = item_grad_full[offset:offset + b, :item_emb_mb.shape[1], :]
        g_text = text_grad_full[offset:offset + b, :text_emb_mb.shape[1], :]

        torch.autograd.backward(
            tensors=[item_emb_mb, text_emb_mb],
            grad_tensors=[g_item.to(item_emb_mb.dtype), g_text.to(text_emb_mb.dtype)],
        )
        offset += b

    return loss.item(), loss_parts


# ==================== Bucle de entrenamiento con GradCache ====================
LEARNING_RATE = 5e-7
EPOCHS        = 20
PATIENCE      = 5

optimizer = torch.optim.Adam(
    list(clip_model.parameters()) + list(core_model.parameters()),
    lr=LEARNING_RATE,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

CKPT_PATH_C4O = POLYVORE_PATH + "checkpoints_clip4outfit/clip4outfit_best_gradcache.pt"
os.makedirs(os.path.dirname(CKPT_PATH_C4O), exist_ok=True)

best_recall5 = -1.0
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    clip_model.train()
    core_model.train()
    train_losses = []

    micro_buffer = []
    pbar = tqdm(train_loader, desc=f"Época {epoch}")
    for batch in pbar:
        item_images, item_mask, input_ids, attention_mask, texts, set_ids = batch
        micro_buffer.append((item_images, item_mask, input_ids, attention_mask, texts))

        if len(micro_buffer) == GRAD_ACCUM_STEPS:
            optimizer.zero_grad()
            loss_val, loss_parts = gradcache_step(clip_model, core_model, micro_buffer, device)
            train_losses.append(loss_val)
            pbar.set_postfix(loss=f"{loss_val:.3f}")

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() == scale_before:
                scheduler.step()

            micro_buffer = []

    # último grupo incompleto de micro-batches al final de la época (si len(train_loader) no es múltiplo de GRAD_ACCUM_STEPS)
    if micro_buffer:
        optimizer.zero_grad()
        loss_val, loss_parts = gradcache_step(clip_model, core_model, micro_buffer, device)
        train_losses.append(loss_val)
        scale_before = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        if scaler.get_scale() == scale_before:
            scheduler.step()

    val_metrics = evaluate_recall_at_k(valid_loader)
    train_loss_mean = sum(train_losses) / len(train_losses)

    print(f"Época {epoch:2d} | loss={train_loss_mean:.4f} | "
          f"val R@1={val_metrics['Recall@1']:.3f} R@5={val_metrics['Recall@5']:.3f} "
          f"R@10={val_metrics['Recall@10']:.3f} R@30={val_metrics['Recall@30']:.3f} "
          f"R@50={val_metrics['Recall@50']:.3f} medRank={val_metrics['MedianRank']:.1f}")

    if val_metrics["Recall@5"] > best_recall5:
        best_recall5 = val_metrics["Recall@5"]
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch,
            "clip_state_dict": clip_model.state_dict(),
            "core_state_dict": core_model.state_dict(),
            "val_recall5": best_recall5,
        }, CKPT_PATH_C4O)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping en época {epoch} (sin mejora en {PATIENCE} épocas)")
            break

print(f"\nMejor Recall@5 en validación: {best_recall5:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Datos extraídos del log de entrenamiento ---
epochs     = list(range(1, 16))
loss_train = [4.7937, 4.2533, 3.9828, 3.7482, 3.5324, 3.3159, 3.1115, 2.9212, 2.7343, 2.5598,
              2.3929, 2.2573, 2.1449, 2.0463, 1.9694]
loss_val   = [4.375, 4.195, 4.113, 4.081, 4.075, 4.070, 4.062, 4.058, 4.055, 4.052,
              4.057, 4.065, 4.078, 4.095, 4.115]
r5_val     = [5.7, 7.1, 7.7, 8.3, 8.5, 8.5, 8.1, 8.6, 8.3, 8.6,
              8.3, 8.5, 8.2, 8.1, 8.2]   # Recall@5 expresado en % (escala 0-100%)

stop_epoch = epochs[-1]                                     # Época 15 (patience=5)
best_epoch = epochs[loss_val.index(min(loss_val))]          # Época 10 (Mínima pérdida de validación)
best_r5    = r5_val[best_epoch - 1]

# --- Configuración del estilo global (Serif + LaTeX aesthetic) ---
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "axes.edgecolor": "0.3",
    "figure.dpi": 300,
})

# Paleta de colores estándar del capítulo
COLOR_TRAIN = "#1f77b4"  # Azul
COLOR_VAL   = "#ff7f0e"  # Naranja
COLOR_STOP  = "#d62728"  # Rojo (Early stopping)

# Crear figura unificada de 2 subplots (1 fila, 2 columnas)
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(11, 4.2))

# =========================================================
# Panel (a): Evolución de las Pérdidas (Train vs Val)
# =========================================================
ax_a.plot(epochs, loss_train, color=COLOR_TRAIN, ls="-", marker="o", ms=4, lw=1.5, label="Train")
ax_a.plot(epochs, loss_val, color=COLOR_VAL, ls="--", marker="s", ms=4, lw=1.5, label="Val")

# Línea vertical roja para Early Stopping
ax_a.axvline(stop_epoch, color=COLOR_STOP, lw=1.1, ls=":", alpha=0.85, label="Early stopping")

ax_a.set_xlabel("Época")
ax_a.set_ylabel(r"Pérdida contrastiva ($\mathcal{L}$)")
ax_a.set_xticks(epochs)
ax_a.set_xlim(0.5, stop_epoch + 0.5)
ax_a.legend(loc="lower left", frameon=True, framealpha=0.9)
ax_a.set_title("(a) Evolución de la pérdida")

# =========================================================
# Panel (b): Recall@5 de validación (%)
# =========================================================
ax_b.plot(epochs, r5_val, color=COLOR_VAL, ls="--", marker="s", ms=4, lw=1.5, label="Val")

# Línea vertical roja para Early Stopping
ax_b.axvline(stop_epoch, color=COLOR_STOP, lw=1.1, ls=":", alpha=0.85, label="Early stopping")

ax_b.set_xlabel("Época")
ax_b.set_ylabel("Recall@5 (%)")
ax_b.set_xticks(epochs)
ax_b.set_xlim(0.5, stop_epoch + 0.5)
ax_b.set_ylim(0.0, 10.0)
ax_b.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax_b.legend(loc="lower left", frameon=True, framealpha=0.9)
ax_b.set_title("(b) Rendimiento en recuperación (Recall@5)")

plt.tight_layout()

# Guardar en formatos de alta calidad
plt.savefig("./fig_entrenamiento_modelo.pdf", bbox_inches="tight")
plt.savefig("./fig_entrenamiento_modelo.png", bbox_inches="tight", dpi=300)
plt.show()

## 9. Evaluación final en test

In [ ]:
CKPT_PATH_C4O = POLYVORE_PATH + "checkpoints_clip4outfit/clip4outfit_best_gradcache.pt"

best_ckpt = torch.load(CKPT_PATH_C4O, map_location=device)
clip_model.load_state_dict(best_ckpt["clip_state_dict"])
core_model.load_state_dict(best_ckpt["core_state_dict"])

test_metrics = evaluate_recall_at_k(test_loader)
print(f"Resultados en test (época {best_ckpt['epoch']}):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.3f}")

In [ ]:
# Uso, tras cargar el mejor checkpoint (igual que en la Sección 9 del notebook):
test_metrics_strict = evaluate_recall_at_k_strict(test_loader)
print("Resultados en test -- criterio ESTRICTO (un solo positivo, como probablemente el paper):")
for k, v in test_metrics_strict.items():
    print(f"  {k}: {v:.3f}")